In [1]:
import pandas as pd
import geopandas as gpd

from shared_utils import rt_utils, catalog_utils, rt_dates, webmap_utils

from calitp_data_analysis import get_fs, geography_utils
from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
gcsgp = GCSGeoPandas()
from segment_speed_utils import helpers, time_series_utils, gtfs_schedule_wrangling, corridor_analysis
from segment_speed_utils.project_vars import SCHED_GCS, SEGMENT_GCS, GTFS_DATA_DICT
import numpy as np
import datetime as dt

ADHOC_PATH = "gs://calitp-analytics-data/data-analyses/rt_segment_speeds/adhoc/"

c:\Users\s156485\Documents\data-analyses\.venv\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.11.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


# Run SCCP/LPP Level of Transit Delay Metric using newer corridor tools

> The speed-based metric for the corridor provided is [insert number] minutes. This is the sum of delays for each trip traversing the corridor as compared to a reference speed of 16 miles per hour. To further explain, we took each corridor trip that we have data for and calculated the hypothetical time it would take for that trip to traverse the corridor at a speed of 16 mph. The difference between the actual time it took for the trip to traverse the corridor and that hypothetical time is the speed-based delay for that trip, and we summed those delays to create the metric for a day ([insert full date]).
>
> This metric is intended to provide a more consistent basis for comparison independent of scheduling practices.
>
> To calculate which routes we consider to be along the corridor, we first filtered to routes stopping within the corridor. We then filtered our data for those trips to the subset of each trip from the last stop before entering the corridor to the first stop after leaving the corridor.
>
> Here is a list of routes included in our analysis: [insert list of routes here]
>
>Additionally, attached is a map of the corridor and transit routes within it.


In [2]:
corridor = gcsgp.read_file(f'{ADHOC_PATH}2026-11-10_sccp_I880InterchangeImprovements.geojson').assign(corridor_id = 1, corridor_distance_meters = 1400)
#  use short side for distance

In [ ]:
analysis_date = rt_dates.DATES['oct2026']

In [34]:
analysis_date

'2026-08-12'

In [35]:
import importlib
importlib.reload(corridor_analysis)

<module 'segment_speed_utils.corridor_analysis' from 'c:\\Users\\s156485\\Documents\\data-analyses\\rt_segment_speeds\\segment_speed_utils\\corridor_analysis.py'>

In [36]:
segment_speeds = corridor_analysis.import_speedmap_segment_speeds(analysis_date)
frequencies = corridor_analysis.get_max_frequencies(segment_speeds)
trip_speeds = corridor_analysis.import_trip_speeds(analysis_date)

2.4 percent of segments have no speed


In [37]:
segment_speeds.name.sort_values().unique()

array(['Alhambra Schedule', 'Antelope Valley Transit Authority Schedule',
       'Arcadia Schedule', 'B-Line Schedule', 'Baldwin Park Schedule',
       'Banning Pass Schedule', 'Basin Transit GMV Schedule',
       'Bay Area 511 AC Transit Schedule', 'Bay Area 511 ACE Schedule',
       'Bay Area 511 Caltrain Schedule',
       'Bay Area 511 Capitol Corridor Schedule',
       'Bay Area 511 County Connection Schedule',
       'Bay Area 511 Dumbarton Express Schedule',
       'Bay Area 511 Emery Go-Round Schedule',
       'Bay Area 511 Golden Gate Transit Schedule',
       'Bay Area 511 MVGO Schedule', 'Bay Area 511 Marin Schedule',
       'Bay Area 511 Mountain View Community Shuttle Schedule',
       'Bay Area 511 Muni Schedule', 'Bay Area 511 PresidiGo Schedule',
       'Bay Area 511 SFO AirTrain Schedule',
       'Bay Area 511 SamTrans Schedule',
       'Bay Area 511 San Francisco Bay Ferry Schedule',
       'Bay Area 511 Santa Clara Transit Schedule',
       'Bay Area 511 SolTrans Sche

In [38]:
# feeds = ["Sacramento Schedule", "Elk Grove Schedule"]
# segment_speeds = segment_speeds.query('name.isin(@feeds)')

In [39]:
corridor_trips = corridor_analysis.find_corridor_data(segment_speeds, corridor, trip_speeds,
                                                    #  custom_relevance_meters=100
                                                    )
corridor_results = corridor_analysis.analyze_corridor_trips(corridor_trips)

0.0 percent of trips with zero seconds
2.5 percent of trips with speeds > 80mph or < 3mph


In [40]:
corridor_trips.route_short_name.unique()

array(['56'], dtype=object)

In [41]:
', '.join(corridor_trips.route_short_name.unique())

'56'

In [46]:
corridor_analysis.validate_corridor_routes(corridor, corridor_trips)

,route_short_name,shape_length,shape_array_key
0,56,790.958024,3c15546db7f183cc66edb5f318a8473e
79,56,761.086195,9ee994c3a958c772d987117539dcf601


2

In [42]:
corridor_improvements = corridor_analysis.analyze_corridor_improvements(corridor_results, **{'trip_mph_floor': 16, 'trip_percent_speedup': 15})

median: 16.18mph... percent mode: 0.15


In [43]:
corridor_improvements.head(3)

,trip_instance_key,corridor_meters,corridor_seconds,corridor_speed_mps,corridor_speed_mph,route_short_name,route_id,shape_array_key,shape_id,schedule_gtfs_dataset_key,time_of_day,corridor_id,improved_corridor_speed_mph,intervention_assumption,improved_corridor_speed_mps,improved_corridor_seconds
0,03134d121fde5f6705df7c4f754cce07,4755.881352,620.0,7.670776,17.159527,56,56,3c15546db7f183cc66edb5f318a8473e,shp-56-54,a6137888630c9c9468fde0c3ba0043be,PM Peak,1,19.733456,15.0% trip speed increase,8.821393,539.130435
1,054a637d241f5a850fcd23818d6d260e,4755.881352,733.0,6.488242,14.514197,56,56,3c15546db7f183cc66edb5f318a8473e,shp-56-54,a6137888630c9c9468fde0c3ba0043be,Midday,1,16.691327,15.0% trip speed increase,7.461478,637.391304
2,0a7bf0ee641dd6b25a53eb380f414d2e,835.068999,85.0,9.824341,21.977051,56,56,3c15546db7f183cc66edb5f318a8473e,shp-56-54,a6137888630c9c9468fde0c3ba0043be,Midday,1,25.273609,15.0% trip speed increase,11.297992,73.913043


In [44]:
summarized = corridor_analysis.summarize_corridor_improvements(corridor_improvements, frequencies, corridor)

c:\Users\s156485\Documents\data-analyses\rt_segment_speeds\segment_speed_utils\corridor_analysis.py:276: FutureWarning: The provided callable <function median at 0x0000022EFDCAE200> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  ).round(1)


In [45]:
summarized.delay_minutes.sum()

54.5